# Two staff labelling comments: Bowker and Stuart-Maxwell

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**McNemar Test** · a worked example from the StatNex teaching guide

**The story.** Two social-media staff members independently labelled the same 262 customer comments as very positive, positive, negative or very negative. Before automating this with AI, the manager wants to know whether humans even agree.

**The question.** When the two staff disagree, does one of them lean towards more positive or more negative labels?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/kappa_coefficient.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Title,Staff A's label,Very positive,Positive,Negative,Very negative,Total
Staff A,Very positive,38.0,9.0,2.0,1.0,50.0
,Positive,12.0,61.0,8.0,3.0,84.0
,Negative,2.0,10.0,52.0,9.0,73.0
,Very negative,1.0,2.0,11.0,41.0,55.0
,Total,53.0,82.0,73.0,54.0,262.0
""")

## Helper: totals

A step that no library does, written out. In the code download it is the file `totals.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import SquareTable, mcnemar

pd.set_option("display.width", 120)

## The data

In [ ]:
# A 2 x 2 table of the same subjects judged twice -- or k x k, with the same k
# categories as rows and columns: rows are one measurement, columns the other.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Very positive", "Positive", "Negative", "Very negative"]
table = data.set_index("Staff A's label")[columns].apply(pd.to_numeric, errors="coerce")
# A row or column named like a total (Total, Sum, or a Japanese word for a total, as listed in totals.py) is a margin, not data,
# so it is left out, as the app does. Spaces, underscores, case and full-width letters
# are ignored when names are compared.
# is_total: defined in the helper cell totals above (totals.py in the code download)

table = table.loc[[not is_total(row) for row in table.index],
                  [not is_total(column) for column in table.columns]]
# A row with a blank count is left out.
table = table.dropna()

## Observed frequency

In [ ]:
observed = pd.concat([table, table.sum(axis=1).rename("Total")], axis=1)
observed = pd.concat([observed, observed.sum().to_frame("Total").T])
print(observed)

## McNemar test

In [ ]:
# Only the two cells where the measurements disagree (b and c) enter the test. With
# the continuity correction, chi-square = (|b - c| - 1)^2 / (b + c) on 1 degree of
# freedom; correction=False would give (b - c)^2 / (b + c).
# How to read it: if the two measurements had the same rate of the first category, a
# case that changed would be as likely to change one way as the other, so b and c would
# be about equal. A small P says the changes lean one way. The correction makes the
# test a little cautious; with b + c below about 25, the exact binomial version
# (b out of b + c, probability 0.5) is the safer choice.
counts = table.to_numpy(float)
if counts.shape == (2, 2):
    result = mcnemar(counts, exact=False, correction=True)
    test = pd.DataFrame([[result.statistic, 1, result.pvalue]], columns=["chi-square", "df", "P"])
else:
    # k x k: Bowker's test that the table is symmetric, summing (n_ij - n_ji)^2 /
    # (n_ij + n_ji) over the pairs either side of the diagonal, on one df per pair that has
    # any cases: an empty pair carries no information. (statsmodels' SquareTable.symmetry
    # keeps k(k-1)/2 df whatever, which makes the test conservative.)
    upper = np.triu_indices(counts.shape[0], 1)
    sums = (counts + counts.T)[upper]
    gaps = (counts - counts.T)[upper]
    statistic = (gaps[sums > 0] ** 2 / sums[sums > 0]).sum()
    df = int((sums > 0).sum())
    test = pd.DataFrame([[statistic, df, stats.chi2.sf(statistic, df)]],
                        columns=["chi-square", "df", "P"])
print()
print(test.to_string(index=False))

if counts.shape != (2, 2):
    # === Marginal homogeneity (Stuart-Maxwell) ===
    # Symmetry implies equal margins but not the reverse, so this asks the weaker
    # question: do the two measurements put the same share of cases in each category?
    result = SquareTable(counts, shift_zeros=False).homogeneity(method="stuart_maxwell")
    homogeneity = pd.DataFrame([[result.statistic, result.df, result.pvalue]],
                               columns=["chi-square", "df", "P"])
    print(homogeneity.to_string(index=False))

## What the result says

These two tests look at the disagreements between the two staff. Bowker's test of symmetry compares each pair of opposite cells. For example, 9 comments are Very positive for staff A and Positive for staff B, and 12 are the other way round. Chi-square is 1.05 with 6 degrees of freedom, P = 0.98.

The Stuart-Maxwell test compares the totals: does one person use some label more often? Staff A used Very positive 50 times and staff B 53 times. Chi-square is 0.36 with 3 degrees of freedom, P = 0.95.

Both P values are far above 0.05. So no lean in either direction is shown: when the staff disagree, the disagreements look balanced. This does not measure how often they agree. Kappa does that.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.